### Imports & Seed

In [1]:
import os
import json
import joblib
import json
import random
import numpy as np
import pandas as pd
from typing import Optional
from typing import Union
from datetime import datetime

from dataclasses import dataclass
from typing import Dict, Any, Tuple, List

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping


def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seeds(42)
print("TensorFlow:", tf.__version__)


TensorFlow: 2.20.0


### Config

In [ ]:
DATA_SOURCE = os.getenv("TRAIN_DATA_SOURCE", "csv").strip().lower()

DATA_PATH = os.getenv("TRAIN_DATA_PATH", "./data/finance_data.csv").strip()
DATE_COL = os.getenv("TRAIN_DATE_COL", "Date").strip()
TARGET_COL = os.getenv("TRAIN_TARGET_COL", "PETR4.SA").strip()

SYMBOL = os.getenv("TRAIN_SYMBOL", "PETR4.SA").strip()
if DATA_SOURCE == "yfinance":
    DATE_COL = "Date"
    TARGET_COL = SYMBOL
    
START_DATE = os.getenv("TRAIN_START_DATE", "2024-01-01").strip()
END_DATE = os.getenv("TRAIN_END_DATE", datetime.today().strftime("%Y-%m-%d")).strip()

def _env_int(name: str, default: int) -> int:
    v = os.getenv(name)
    if v is None or str(v).strip() == "":
        return default
    try:
        return int(float(str(v).strip()))
    except Exception:
        return default

TRAIN_MAX_ROWS = _env_int("TRAIN_MAX_ROWS", 0)

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

TRAIN_GRID_MODE = os.getenv("TRAIN_GRID_MODE", "full").strip().lower()

if TRAIN_GRID_MODE in ("fast", "quick", "small"):
    HYPERPARAM_GRID = {
        "lstm_units": [32],
        "dropout_rate": [0.2],
        "batch_size": [32],
        "epochs": [10],
        "window_size": [30],
        "learning_rate": [1e-3],
    }
elif TRAIN_GRID_MODE in ("medium", "mid"):
    HYPERPARAM_GRID = {
        "lstm_units": [32, 64],
        "dropout_rate": [0.2],
        "batch_size": [32],
        "epochs": [20],
        "window_size": [30, 60],
        "learning_rate": [1e-3],
    }
else:
    HYPERPARAM_GRID = {
        "lstm_units": [32, 64, 96],
        "dropout_rate": [0.2, 0.3],
        "batch_size": [16, 32],
        "epochs": [30, 50],
        "window_size": [30, 60],
        "learning_rate": [1e-3],
    }

EARLY_STOPPING = dict(
    monitor="val_loss",
    patience=_env_int("TRAIN_EARLY_STOPPING_PATIENCE", 10),
    restore_best_weights=True
)

ARTIFACT_PATH = os.getenv("TRAIN_ARTIFACT_PATH", "best_lstm_artifact.pkl").strip()

OUTPUT_PATH = os.getenv("TRAIN_OUTPUT_PATH", "./prediction_output.json").strip()

print("Config:", {"DATA_SOURCE": DATA_SOURCE, "TARGET_COL": TARGET_COL, "TRAIN_MAX_ROWS": TRAIN_MAX_ROWS, "TRAIN_GRID_MODE": TRAIN_GRID_MODE})

### Load data

In [ ]:
def load_data() -> pd.DataFrame:
    if DATA_SOURCE == "csv":
        df = pd.read_csv(DATA_PATH)
        if TRAIN_MAX_ROWS and TRAIN_MAX_ROWS > 0:
            df = df.tail(TRAIN_MAX_ROWS).reset_index(drop=True)
        return df

    if DATA_SOURCE == "yfinance":
        import yfinance as yf
        df = yf.download(SYMBOL, start=START_DATE, end=END_DATE, progress=False)
        df = df.reset_index()  # cria coluna Date
        if TRAIN_MAX_ROWS and TRAIN_MAX_ROWS > 0:
            df = df.tail(TRAIN_MAX_ROWS).reset_index(drop=True)
        return df

    raise ValueError(f"DATA_SOURCE inválido: {DATA_SOURCE}. Use 'csv' ou 'yfinance'.")

df = load_data()
df.head()

,Date,PETR4.SA
0,2024-01-02,27.692333
1,2024-01-03,28.557262
2,2024-01-04,28.315382
3,2024-01-05,28.381353
4,2024-01-08,28.168781


### Prepare series

In [4]:
def prepare_series(
    df: pd.DataFrame,
    date_col: str,
    target_col: str
) -> pd.Series:
    df = df.copy()

    if date_col not in df.columns:
        raise ValueError(f"Coluna de data '{date_col}' não encontrada no DataFrame.")

    if target_col not in df.columns:
        raise ValueError(f"Coluna alvo '{target_col}' não encontrada no DataFrame.")

    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col, target_col])

    df = df.sort_values(date_col).drop_duplicates(subset=[date_col], keep="last")
    series = df[target_col].astype(float)

    series.index = df[date_col].values
    series.name = target_col
    return series

series = prepare_series(df_raw, DATE_COL, TARGET_COL)
series.head(), series.tail()


(2024-01-02    27.692333
 2024-01-03    28.557262
 2024-01-04    28.315382
 2024-01-05    28.381353
 2024-01-08    28.168781
 Name: PETR4.SA, dtype: float64,
 2025-12-29    30.730000
 2025-12-30    30.820000
 2026-01-02    30.709999
 2026-01-05    30.200001
 2026-01-06    29.639999
 Name: PETR4.SA, dtype: float64)

### Temporal split (train/val/test) sem vazamento

In [5]:
def temporal_split(
    series: pd.Series,
    train_ratio: float,
    val_ratio: float,
    test_ratio: float
) -> Tuple[pd.Series, pd.Series, pd.Series]:
    n = len(series)
    if n < 10:
        raise ValueError("Série muito curta para split/treino.")

    if not np.isclose(train_ratio + val_ratio + test_ratio, 1.0):
        raise ValueError("Ratios devem somar 1.0")

    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    train = series.iloc[:train_end]
    val   = series.iloc[train_end:val_end]
    test  = series.iloc[val_end:]
    return train, val, test

train_s, val_s, test_s = temporal_split(series, TRAIN_RATIO, VAL_RATIO, TEST_RATIO)
len(train_s), len(val_s), len(test_s)


(352, 76, 76)

### Scaling (fit só no treino)

In [6]:
def fit_scaler_on_train(train: pd.Series) -> MinMaxScaler:
    scaler = MinMaxScaler()
    scaler.fit(train.values.reshape(-1, 1))
    return scaler

def scale_series(s: pd.Series, scaler: MinMaxScaler) -> np.ndarray:
    return scaler.transform(s.values.reshape(-1, 1)).reshape(-1)

scaler = fit_scaler_on_train(train_s)

train_scaled = scale_series(train_s, scaler)
val_scaled   = scale_series(val_s, scaler)
test_scaled  = scale_series(test_s, scaler)

train_scaled[:3], val_scaled[:3], test_scaled[:3]


(array([0.184762  , 0.28711909, 0.25849458]),
 array([0.27337802, 0.29298439, 0.29401277]),
 array([0.5257536 , 0.54755644, 0.51198318]))

### Windowing (após normalizar) com contexto correto

In [7]:
def create_temporal_window_1d(values_1d: np.ndarray, window_size: int) -> Tuple[np.ndarray, np.ndarray]:
    """
    values_1d: array 1D já normalizado.
    Retorna X: (samples, window_size, 1) e y: (samples, 1)
    """
    X, y = [], []
    for i in range(window_size, len(values_1d)):
        X.append(values_1d[i - window_size:i])
        y.append(values_1d[i])
    X = np.array(X).reshape(-1, window_size, 1)
    y = np.array(y).reshape(-1, 1)
    return X, y

def make_datasets(
    train_scaled: np.ndarray,
    val_scaled: np.ndarray,
    test_scaled: np.ndarray,
    window_size: int
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    X_train, y_train = create_temporal_window_1d(train_scaled, window_size)

    # val: precisa do "contexto" do final do treino para prever o início da validação
    val_concat = np.concatenate([train_scaled[-window_size:], val_scaled])
    X_val, y_val = create_temporal_window_1d(val_concat, window_size)  # y_val corresponde a val_scaled inteiro

    test_context = np.concatenate([train_scaled, val_scaled])[-window_size:]
    test_concat = np.concatenate([test_context, test_scaled])
    X_test, y_test = create_temporal_window_1d(test_concat, window_size)

    return X_train, y_train, X_val, y_val, X_test, y_test

X_train, y_train, X_val, y_val, X_test, y_test = make_datasets(train_scaled, val_scaled, test_scaled, window_size=HYPERPARAM_GRID["window_size"][0])
X_train.shape, X_val.shape, X_test.shape


((322, 30, 1), (76, 30, 1), (76, 30, 1))

### Model factory

In [8]:
def build_lstm_model(
    window_size: int,
    lstm_units: int,
    dropout_rate: float,
    learning_rate: float = 1e-3
) -> tf.keras.Model:
    model = Sequential([
        LSTM(lstm_units, return_sequences=True, input_shape=(window_size, 1)),
        Dropout(dropout_rate),
        LSTM(max(lstm_units // 2, 8)),
        Dropout(dropout_rate),
        Dense(1)
    ])

    opt = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=opt, loss="mse")
    return model


### Grid Search (simple loop) + métricas (MAPE desnormalizado)

In [9]:
def inverse_1d(scaler: MinMaxScaler, arr_2d: np.ndarray) -> np.ndarray:
    """
    arr_2d: shape (n,1) normalizado
    retorna shape (n,) em valores reais
    """
    return scaler.inverse_transform(arr_2d).reshape(-1)

def eval_regression_real(y_true_real: np.ndarray, y_pred_real: np.ndarray) -> Dict[str, float]:
    mae = mean_absolute_error(y_true_real, y_pred_real)
    rmse = np.sqrt(mean_squared_error(y_true_real, y_pred_real))
    mape = mean_absolute_percentage_error(y_true_real, y_pred_real)
    return {"mae": float(mae), "rmse": float(rmse), "mape": float(mape)}

results: List[Dict[str, Any]] = []

# loop simples (sem KerasTuner)
for window_size in HYPERPARAM_GRID["window_size"]:
    X_train, y_train, X_val, y_val, _, _ = make_datasets(train_scaled, val_scaled, test_scaled, window_size=window_size)

    for lstm_units in HYPERPARAM_GRID["lstm_units"]:
        for dropout_rate in HYPERPARAM_GRID["dropout_rate"]:
            for batch_size in HYPERPARAM_GRID["batch_size"]:
                for epochs in HYPERPARAM_GRID["epochs"]:
                    for lr in HYPERPARAM_GRID.get("learning_rate", [1e-3]):

                        tf.keras.backend.clear_session()

                        model = build_lstm_model(
                            window_size=window_size,
                            lstm_units=lstm_units,
                            dropout_rate=dropout_rate,
                            learning_rate=lr
                        )

                        es = EarlyStopping(**EARLY_STOPPING)

                        history = model.fit(
                            X_train, y_train,
                            validation_data=(X_val, y_val),
                            epochs=epochs,
                            batch_size=batch_size,
                            callbacks=[es],
                            shuffle=False,
                            verbose=0
                        )

                        # Predição na validação
                        y_val_pred = model.predict(X_val, verbose=0)

                        # Desnormaliza (MAPE em valores reais)
                        y_val_real = inverse_1d(scaler, y_val)
                        y_pred_real = inverse_1d(scaler, y_val_pred)

                        metrics = eval_regression_real(y_val_real, y_pred_real)

                        results.append({
                            "window_size": window_size,
                            "lstm_units": lstm_units,
                            "dropout_rate": dropout_rate,
                            "batch_size": batch_size,
                            "epochs": epochs,
                            "learning_rate": lr,
                            "val_mae": metrics["mae"],
                            "val_rmse": metrics["rmse"],
                            "val_mape": metrics["mape"],
                            "best_epoch": len(history.history["loss"])
                        })

results_df = pd.DataFrame(results).sort_values("val_mape", ascending=True)
results_df.head(10)


/Users/marina.oliveira/Documents/ESTUDOS/tc-financas/.venv/lib/python3.9/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/Users/marina.oliveira/Documents/ESTUDOS/tc-financas/.venv/lib/python3.9/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/Users/marina.oliveira/Documents/ESTUDOS/tc-financas/.venv/lib/python3.9/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


/Users/marina.oliveira/Documents/ESTUDOS/tc-financas/.venv/lib/python3.9/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/Users/marina.oliveira/Documents/ESTUDOS/tc-financas/.venv/lib/python3.9/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/Users/marina.oliveira/Documents/ESTUDOS/tc-financas/.venv/lib/python3.9/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/

,window_size,lstm_units,dropout_rate,batch_size,epochs,learning_rate,val_mae,val_rmse,val_mape,best_epoch
17,30,96,0.2,16,50,0.001,0.388382,0.538375,0.013077,50
9,30,64,0.2,16,50,0.001,0.392905,0.539059,0.013217,50
41,60,96,0.2,16,50,0.001,0.393440,0.537470,0.013237,50
45,60,96,0.3,16,50,0.001,0.398054,0.532365,0.013409,50
33,60,64,0.2,16,50,0.001,0.411343,0.546183,0.013844,50
13,30,64,0.3,16,50,0.001,0.413408,0.560390,0.013891,50
37,60,64,0.3,16,50,0.001,0.414149,0.546715,0.013924,50
21,30,96,0.3,16,50,0.001,0.419666,0.564877,0.014109,43
43,60,96,0.2,32,50,0.001,0.432468,0.571925,0.014540,50
23,30,96,0.3,32,50,0.001,0.443529,0.591670,0.014900,50


### Best params + Treino final do zero

In [10]:
best = results_df.iloc[0].to_dict()
best_params = {
    "window_size": int(best["window_size"]),
    "lstm_units": int(best["lstm_units"]),
    "dropout_rate": float(best["dropout_rate"]),
    "batch_size": int(best["batch_size"]),
    "epochs": int(best["epochs"]),
    "learning_rate": float(best["learning_rate"]),
    "best_epoch": int(best["best_epoch"]),
}

best_params


{'window_size': 30,
 'lstm_units': 96,
 'dropout_rate': 0.2,
 'batch_size': 16,
 'epochs': 50,
 'learning_rate': 0.001,
 'best_epoch': 50}

In [11]:
# Recria datasets com a melhor janela
X_train, y_train, X_val, y_val, X_test, y_test = make_datasets(
    train_scaled, val_scaled, test_scaled,
    window_size=best_params["window_size"]
)

tf.keras.backend.clear_session()

final_model = build_lstm_model(
    window_size=best_params["window_size"],
    lstm_units=best_params["lstm_units"],
    dropout_rate=best_params["dropout_rate"],
    learning_rate=best_params["learning_rate"]
)

es = EarlyStopping(**EARLY_STOPPING)

history = final_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=best_params["epochs"],
    batch_size=best_params["batch_size"],
    callbacks=[es],
    shuffle=False,
    verbose=0
)

print("Final trained epochs:", len(history.history["loss"]))


/Users/marina.oliveira/Documents/ESTUDOS/tc-financas/.venv/lib/python3.9/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Final trained epochs: 47


### Evaluate on test (MAPE desnormalizado)

In [12]:
y_test_pred = final_model.predict(X_test, verbose=0)

y_test_real = inverse_1d(scaler, y_test)
y_pred_real = inverse_1d(scaler, y_test_pred)

test_metrics = eval_regression_real(y_test_real, y_pred_real)
test_metrics


{'mae': 0.3721278843126803,
 'rmse': 0.4735274490126883,
 'mape': 0.012265821952738523}

### Save best artifact (.pkl)

In [13]:
def save_artifact_pkl(
    path: str,
    model: tf.keras.Model,
    scaler: MinMaxScaler,
    window_size: int,
    feature_names: List[str],
    best_params: Dict[str, Any],
    extra_metadata: Optional[Dict[str, Any]] = None
) -> None:
    artifact = {
        "scaler": scaler,
        "window_size": int(window_size),
        "feature_names": feature_names,
        "model_json": model.to_json(),
        "model_weights": model.get_weights(),
        "best_params": best_params,
        "compile": {
            "loss": "mse",
            "optimizer": "adam",
        },
        "metadata": {
            "tf_version": tf.__version__,
            **(extra_metadata or {})
        }
    }
    joblib.dump(artifact, path)

save_artifact_pkl(
    path=ARTIFACT_PATH,
    model=final_model,
    scaler=scaler,
    window_size=best_params["window_size"],
    feature_names=[TARGET_COL],
    best_params=best_params,
    extra_metadata={"target_col": TARGET_COL}
)

print("Saved:", ARTIFACT_PATH)

Saved: best_lstm_artifact.pkl


### Load artifact + Predict (inference)

In [16]:
def load_artifact_pkl(path: str) -> Dict[str, Any]:
    artifact = joblib.load(path)

    model = tf.keras.models.model_from_json(artifact["model_json"])
    model.set_weights(artifact["model_weights"])

    model.compile(optimizer="adam", loss="mse")

    artifact["model"] = model
    return artifact

def predict_next_from_recent(
    artifact: Dict[str, Any],
    recent_values: Union[pd.Series, np.ndarray]
) -> float:
    """
    Recebe os últimos valores REAIS (não normalizados).
    Precisa ter pelo menos window_size pontos.
    Retorna previsão do próximo valor REAL.
    """
    scaler: MinMaxScaler = artifact["scaler"]
    window_size: int = artifact["window_size"]
    model: tf.keras.Model = artifact["model"]

    if isinstance(recent_values, pd.Series):
        recent = recent_values.values.astype(float)
    else:
        recent = np.asarray(recent_values, dtype=float)

    if len(recent) < window_size:
        raise ValueError(f"Precisa de pelo menos {window_size} valores recentes.")

    last_window = recent[-window_size:].reshape(-1, 1)
    last_window_scaled = scaler.transform(last_window).reshape(1, window_size, 1)

    pred_scaled = model.predict(last_window_scaled, verbose=0).reshape(-1, 1)
    pred_real = scaler.inverse_transform(pred_scaled).reshape(-1)[0]
    return float(pred_real)


artifact = load_artifact_pkl(ARTIFACT_PATH)

# usa os últimos pontos do TESTE real como exemplo de "recent_values"
example_recent = test_s.iloc[-artifact["window_size"]:]
next_pred = predict_next_from_recent(artifact, example_recent)

next_pred

with open(OUTPUT_PATH, "w") as f:
    json.dump({"prediction_r$": next_pred}, f)
print(f"Salvo em {OUTPUT_PATH}")

Salvo em ./prediction_output.json
